# Phase 1 EDA: Kaggle CNC Mill Tool Wear

## 목적
- 18개 실험의 센서 분포를 시각적으로 탐색
- worn(마모) vs unworn(정상) 차이 확인
- 가공 단계별 센서 패턴 파악
- 42개 유효 컬럼 중 이상탐지에 유용한 센서 식별
- timestamp / equipment_id 합성 규칙 설계 (Open Item #1)

## 데이터 개요
- **출처:** Michigan SMART Lab (Kaggle)
- **구조:** 18개 실험 CSV, 각 실험 = 1회 CNC 밀링 가공
- **컬럼:** 48개 (유효 42개), 100ms 샘플링
- **라벨:** worn 10실험 / unworn 8실험 (train.csv)

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

# 프로젝트 루트 기준으로 Kaggle CNC Mill 데이터 경로를 잡습니다.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "raw" / "kaggle-cnc-mill"
TRAIN_PATH = DATA_DIR / "train.csv"
EXPERIMENT_PATHS = sorted(DATA_DIR.glob("experiment_*.csv"))

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DATA_DIR exists: {DATA_DIR.exists()}")
print(f"실험 파일 수: {len(EXPERIMENT_PATHS)}")
print(f"train.csv exists: {TRAIN_PATH.exists()}")

PROJECT_ROOT: c:\Users\nugikim\Desktop\workspace\Manufacturing-ax-agent
DATA_DIR exists: True
실험 파일 수: 18
train.csv exists: True


In [2]:
# train.csv는 실험별 메타데이터와 라벨을 담고 있습니다.
train_df = pd.read_csv(TRAIN_PATH)
train_df = train_df.rename(columns={"No": "experiment_no"})
train_df["experiment_id"] = train_df["experiment_no"].map(lambda value: f"experiment_{value:02d}")
train_df["equipment_id"] = train_df["experiment_no"].map(lambda value: f"CNC-{value:03d}")

print("train.csv shape:", train_df.shape)
display(train_df.head())
display(train_df["tool_condition"].value_counts(dropna=False).rename("count").to_frame())
display(train_df.groupby(["tool_condition", "machining_finalized"]).size().rename("count").reset_index())

train.csv shape: (18, 9)


,experiment_no,material,feedrate,clamp_pressure,tool_condition,machining_finalized,passed_visual_inspection,experiment_id,equipment_id
0,1,wax,6,4.0,unworn,yes,yes,experiment_01,CNC-001
1,2,wax,20,4.0,unworn,yes,yes,experiment_02,CNC-002
2,3,wax,6,3.0,unworn,yes,yes,experiment_03,CNC-003
3,4,wax,6,2.5,unworn,no,NaN,experiment_04,CNC-004
4,5,wax,20,3.0,unworn,no,NaN,experiment_05,CNC-005


,count
tool_condition,
worn,10
unworn,8


,tool_condition,machining_finalized,count
0,unworn,no,2
1,unworn,yes,6
2,worn,no,2
3,worn,yes,8


In [3]:
# 각 실험 CSV의 크기와 컬럼 일관성을 먼저 확인합니다.
experiment_summaries = []
column_sets = []

for path in EXPERIMENT_PATHS:
    experiment_df = pd.read_csv(path)
    column_sets.append(tuple(experiment_df.columns))
    experiment_summaries.append(
        {
            "experiment_id": path.stem,
            "rows": len(experiment_df),
            "cols": len(experiment_df.columns),
            "first_process": experiment_df["Machining_Process"].iloc[0],
            "last_process": experiment_df["Machining_Process"].iloc[-1],
        }
    )

summary_df = pd.DataFrame(experiment_summaries).sort_values("experiment_id")
print("컬럼 구조가 완전히 같은 실험 파일 수:", len(set(column_sets)))
display(summary_df)
display(summary_df[["rows", "cols"]].describe().T)

컬럼 구조가 완전히 같은 실험 파일 수: 1


,experiment_id,rows,cols,first_process,last_process
0,experiment_01,1055,48,Starting,end
1,experiment_02,1668,48,Prep,End
2,experiment_03,1521,48,Prep,End
3,experiment_04,532,48,Prep,End
4,experiment_05,462,48,Prep,End
5,experiment_06,1296,48,Prep,End
6,experiment_07,565,48,Prep,End
7,experiment_08,605,48,Prep,End
8,experiment_09,740,48,Prep,End
9,experiment_10,1301,48,Prep,End


,count,mean,std,min,25%,50%,75%,max
rows,18.0,1404.777778,716.050159,462.0,638.75,1341.0,2212.25,2332.0
cols,18.0,48.000000,0.000000,48.0,48.00,48.0,48.00,48.0


In [4]:
# 모든 실험을 합쳐 상수 컬럼과 0값 고정 컬럼 후보를 찾습니다.
all_experiments = []

for path in EXPERIMENT_PATHS:
    experiment_id = path.stem
    experiment_no = int(experiment_id.split("_")[-1])
    experiment_df = pd.read_csv(path).copy()
    experiment_df["experiment_id"] = experiment_id
    experiment_df["equipment_id"] = f"CNC-{experiment_no:03d}"
    experiment_df["sequence"] = range(len(experiment_df))
    experiment_df["timestamp"] = pd.Timestamp("2026-01-01 00:00:00") + pd.to_timedelta(experiment_df["sequence"] * 100, unit="ms")
    all_experiments.append(experiment_df)

full_df = pd.concat(all_experiments, ignore_index=True)
nunique_series = full_df.nunique(dropna=False).sort_values()
constant_columns = nunique_series[nunique_series == 1].index.tolist()
zero_only_columns = [column for column in full_df.columns if pd.api.types.is_numeric_dtype(full_df[column]) and full_df[column].fillna(0).eq(0).all()]

print("통합 데이터 shape:", full_df.shape)
print("상수 컬럼 수:", len(constant_columns))
print("0값 고정 numeric 컬럼 수:", len(zero_only_columns))
display(pd.DataFrame({"constant_columns": pd.Series(constant_columns)}))
display(pd.DataFrame({"zero_only_columns": pd.Series(zero_only_columns)}))
display(nunique_series.head(15).rename("nunique").to_frame())

통합 데이터 shape: (25286, 52)
상수 컬럼 수: 5
0값 고정 numeric 컬럼 수: 4


,constant_columns
0,Z1_OutputVoltage
1,Z1_OutputCurrent
2,Z1_DCBusVoltage
3,Z1_CurrentFeedback
4,S1_SystemInertia


,zero_only_columns
0,Z1_CurrentFeedback
1,Z1_DCBusVoltage
2,Z1_OutputCurrent
3,Z1_OutputVoltage


,nunique
Z1_OutputVoltage,1
Z1_OutputCurrent,1
Z1_DCBusVoltage,1
Z1_CurrentFeedback,1
S1_SystemInertia,1
M1_CURRENT_PROGRAM_NUMBER,3
S1_CommandAcceleration,5
M1_CURRENT_FEEDRATE,6
Machining_Process,11
X1_OutputCurrent,12


In [5]:
# timestamp / equipment_id 합성 규칙 초안을 눈으로 확인합니다.
canonical_preview_columns = [
    "experiment_id",
    "equipment_id",
    "sequence",
    "timestamp",
    "Machining_Process",
]

sensor_preview_columns = [column for column in full_df.columns if column not in canonical_preview_columns][:5]
preview_columns = canonical_preview_columns + sensor_preview_columns

display(full_df[preview_columns].head(10))
display(
    full_df.groupby("experiment_id")["timestamp"]
    .agg(start="min", end="max", rows="count")
    .reset_index()
    .head()
)

,experiment_id,equipment_id,sequence,timestamp,Machining_Process,X1_ActualPosition,X1_ActualVelocity,X1_ActualAcceleration,X1_CommandPosition,X1_CommandVelocity
0,experiment_01,CNC-001,0,2026-01-01 00:00:00.000,Starting,198.0,0.0,0.00,198.0,0.0
1,experiment_01,CNC-001,1,2026-01-01 00:00:00.100,Prep,198.0,-10.8,-350.00,198.0,-13.6
2,experiment_01,CNC-001,2,2026-01-01 00:00:00.200,Prep,196.0,-17.8,-6.25,196.0,-17.9
3,experiment_01,CNC-001,3,2026-01-01 00:00:00.300,Prep,194.0,-18.0,0.00,194.0,-17.9
4,experiment_01,CNC-001,4,2026-01-01 00:00:00.400,Prep,193.0,-17.9,-18.80,192.0,-17.9
5,experiment_01,CNC-001,5,2026-01-01 00:00:00.500,Prep,191.0,-17.6,81.20,191.0,-17.9
6,experiment_01,CNC-001,6,2026-01-01 00:00:00.600,Prep,189.0,-17.9,-6.25,189.0,-17.9
7,experiment_01,CNC-001,7,2026-01-01 00:00:00.700,Prep,187.0,-17.8,50.00,187.0,-17.9
8,experiment_01,CNC-001,8,2026-01-01 00:00:00.800,Prep,185.0,-17.9,25.00,185.0,-17.9
9,experiment_01,CNC-001,9,2026-01-01 00:00:00.900,Prep,184.0,-17.9,12.50,183.0,-17.9


,experiment_id,start,end,rows
0,experiment_01,2026-01-01,2026-01-01 00:01:45.400,1055
1,experiment_02,2026-01-01,2026-01-01 00:02:46.700,1668
2,experiment_03,2026-01-01,2026-01-01 00:02:32.000,1521
3,experiment_04,2026-01-01,2026-01-01 00:00:53.100,532
4,experiment_05,2026-01-01,2026-01-01 00:00:46.100,462
